In [0]:
from pyspark.sql.functions import col, explode
from delta.tables import DeltaTable

In [0]:
#Leemos la tabla de la anterior capa
df_bronze = spark.table("workspace.default.bronze_fpl_data")

In [0]:
# Buscamos extraer y aplanar dicho array, nos interesan los objetos de "elements" 
# Con la funcion explode se puede convertir cada elemento de dicho array en una fila
df_players_exploded = df_bronze.select(explode(col("elements")).alias("player"))

In [0]:
# Extraemos los atributos que queramos y los renombramos a un formato mas estandar
df_silver_players = df_players_exploded.select(
    col("player.id").alias("player_id"),
    col("player.first_name").alias("first_name"),
    col("player.second_name").alias("last_name"),
    col("player.web_name").alias("known_name"),
    col("player.team").alias("team_id"),
    col("player.element_type").alias("position_id"),
    (col("player.now_cost") / 10).alias("current_price_millions"), # La API da el precio multiplicado por 10
    col("player.total_points").alias("total_points"),
    col("player.minutes").alias("minutes_played"),
    col("player.goals_scored").alias("goals_scored"),
    col("player.assists").alias("assists"),
    col("player.status").alias("injury_status")
)

In [0]:
table_name = "workspace.default.silver_fpl_players"

# Comprobamos si ya existe la tabla para crearla de cero o actualizar la existente
if spark.catalog.tableExists(table_name):
    #Usamos MERGE si ya existía previamente
    delta_table = DeltaTable.forName(spark, table_name)
    
    (delta_table.alias("target")
     .merge(
         df_silver_players.alias("source"),
         "target.player_id = source.player_id" 
     )
     .whenMatchedUpdateAll()   #Para actualizar la fila
     .whenNotMatchedInsertAll() #Para crear la fila
     .execute()
    )
    print("Tabla ACTUALIZADA")
else:
    #Creamos la tabla si no existía 
    df_silver_players.write.format("delta").saveAsTable(table_name)
    print("Tabla base de jugadores creada.")

In [0]:
display(df_silver_players)

In [0]:
#Vamos a crear dos tablas de dimensiones para poder realizar un STAR SCHEMA en caso de que lo necesitemos
df_teams = df_bronze.select(explode(col("teams")).alias("team")).select(
    col("team.id").alias("team_id"),
    col("team.name").alias("team_name"),
    col("team.short_name").alias("team_short_name")
)
df_teams.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_fpl_teams")

df_positions = df_bronze.select(explode(col("element_types")).alias("pos")).select(
    col("pos.id").alias("position_id"),
    col("pos.singular_name").alias("position_name")
)
df_positions.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_fpl_positions")

In [0]:
display(df_teams)

In [0]:
display(df_positions)